# contacts-v1 — explore generated documents

Loads the `docs.parquet` produced by `contacts-v1 generate` (one row per
protein: the `document` token string plus metadata columns) and gives a few
starting points for a closer look — metadata distributions, a document
decoder, and per-protein inspection.

See [`README.md`](README.md) / [`SPEC.md`](SPEC.md) for the document format.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq

# Path to your generated documents parquet — change as needed.
DOCS_PARQUET = Path("/home/bizon/contacts_v1_shard0/docs.parquet")

df = pq.read_table(DOCS_PARQUET).to_pandas()
print(f"{len(df)} documents  |  columns:")
print(list(df.columns))
df.head(3)

## Metadata overview

In [ ]:
metric_cols = [
    "seq_len", "global_plddt",
    "contacts_pre_filter", "contacts_passing_min_degree",
    "contacts_emitted", "contacts_excluded",
    "highest_contact_degree", "lowest_included_contact_degree",
    "num_tokens",
]
df[metric_cols].describe().T

In [ ]:
n = len(df)
print(f"documents:                     {n}")
print(f"truncated (hit 8192 budget):   {int(df.truncated.sum())} ({100 * df.truncated.mean():.1f}%)")
print(f"within 192 tokens of budget:   {int((df.num_tokens >= 8000).sum())}")
filtered = 1 - df.contacts_passing_min_degree.sum() / df.contacts_pre_filter.sum()
print(f"dropped by min-degree filter:  {100 * filtered:.1f}% of all found contacts")
print(f"total contacts emitted:        {int(df.contacts_emitted.sum()):,}")

## Distributions

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
df.seq_len.hist(bins=40, ax=ax[0, 0]); ax[0, 0].set_title("seq_len (residues)")
df.num_tokens.hist(bins=40, ax=ax[0, 1]); ax[0, 1].set_title("num_tokens")
ax[0, 1].axvline(8192, color="r", ls="--", lw=1)
df.contacts_emitted.hist(bins=40, ax=ax[1, 0]); ax[1, 0].set_title("contacts_emitted")
ax[1, 1].scatter(df.seq_len, df.num_tokens, s=6, alpha=0.25)
ax[1, 1].axhline(8192, color="r", ls="--", lw=1)
ax[1, 1].set_xlabel("seq_len"); ax[1, 1].set_ylabel("num_tokens")
ax[1, 1].set_title("length vs tokens")
fig.tight_layout()

## Decode a document

Each `document` is one token string:
`<contacts-v1> <begin_sequence> … <begin_statements> … <end>`. The helpers
below recover the residues (`<pN> <AA>`), the N/C termini, the contacts
(`<contact> <pX> <pY>`), and the chain sequence in n-term→c-term order.

Positions are randomized wrap-around indices, so a contact's `<pX>`/`<pY>`
are *document* indices, not sequence positions — use `pos_to_seq` to map
them back to 0-based sequence positions.

In [ ]:
_AA3_TO_1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q",
    "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K",
    "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
    "TYR": "Y", "VAL": "V", "UNK": "X",
}
NUM_INDICES = 2000


def parse_document(doc):
    """Decode a document into (n_term, c_term, residues, contacts).

    residues: {pos_index: "AAA"};  contacts: [(pos_x, pos_y), ...].
    """
    toks = doc.split()
    bs = toks.index("<begin_sequence>")
    bt = toks.index("<begin_statements>")
    end = toks.index("<end>")
    residues, n_term, c_term = {}, None, None
    seq = toks[bs + 1:bt]
    for i in range(0, len(seq), 2):
        a, b = seq[i], seq[i + 1]
        if a == "<n-term>":
            n_term = int(b[2:-1])
        elif a == "<c-term>":
            c_term = int(b[2:-1])
        else:
            residues[int(a[2:-1])] = b.strip("<>")
    contacts = []
    st = toks[bt + 1:end]
    for i in range(0, len(st), 3):
        contacts.append((int(st[i + 1][2:-1]), int(st[i + 2][2:-1])))
    return n_term, c_term, residues, contacts


def ordered_positions(n_term, c_term, num_indices=NUM_INDICES):
    """Position indices from n-term to c-term (wrapping around)."""
    out, pos = [], n_term
    while True:
        out.append(pos)
        if pos == c_term:
            break
        pos = (pos + 1) % num_indices
    return out


def sequence_1letter(n_term, c_term, residues):
    return "".join(
        _AA3_TO_1.get(residues[p], "X")
        for p in ordered_positions(n_term, c_term)
    )


def pos_to_seq(n_term, c_term, num_indices=NUM_INDICES):
    """Map a document position index -> 0-based sequence position."""
    return {p: k for k, p in enumerate(ordered_positions(n_term, c_term, num_indices))}

In [ ]:
row = df.iloc[0]
n_term, c_term, residues, contacts = parse_document(row.document)
p2s = pos_to_seq(n_term, c_term)

print(row.entry_id, "| seq_len", row.seq_len, "| plddt", round(row.global_plddt, 1),
      "| contacts", row.contacts_emitted, "| truncated", row.truncated)
print(f"n_term <p{n_term}>   c_term <p{c_term}>")
print("sequence:", sequence_1letter(n_term, c_term, residues))
print("\nfirst 10 contacts (mapped to sequence positions):")
for x, y in contacts[:10]:
    i, j = p2s[x], p2s[y]
    print(f"  <p{x}>/<p{y}>  ->  seq {i}-{j}  ({residues[x]}{i}-{residues[y]}{j})")

## Contact-map heatmaps

Each document encodes a binary contact map — which residue pairs are
listed as contacts (after the min-degree filter and truncation). Below is
the contact map for the first `N` documents: a symmetric `L x L` grid
where a dark cell at `(i, j)` means residues `i` and `j` are in contact.

In [ ]:
def contact_matrix(document):
    """L x L binary symmetric contact map, indexed by sequence position."""
    n_term, c_term, residues, contacts = parse_document(document)
    p2s = pos_to_seq(n_term, c_term)
    L = len(residues)
    m = np.zeros((L, L), dtype=np.uint8)
    for x, y in contacts:
        i, j = p2s[x], p2s[y]
        m[i, j] = m[j, i] = 1
    return m


N = 10  # number of documents to show
ncols = 5
nrows = (N + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.4 * nrows))
for k, ax in enumerate(np.atleast_1d(axes).flatten()):
    if k >= N:
        ax.axis("off")
        continue
    row = df.iloc[k]
    ax.imshow(contact_matrix(row.document), cmap="Greys",
              origin="upper", interpolation="nearest")
    ax.set_title(f"{row.entry_id}\nL={row.seq_len} · {row.contacts_emitted} contacts",
                 fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f"contacts-v1 contact maps — first {N} documents", fontsize=12)
fig.tight_layout()

## Pick a specific protein

In [ ]:
# the largest protein, and the ones that hit the token budget
largest = df.loc[df.seq_len.idxmax()]
print("largest:", largest.entry_id, "| len", largest.seq_len, "| tokens", largest.num_tokens)

truncated = df[df.truncated]
print("truncated docs:", len(truncated))
truncated[["entry_id", "seq_len", "contacts_pre_filter",
           "contacts_emitted", "contacts_excluded", "num_tokens"]]

In [ ]:
# look up one by entry_id and decode it
entry = "AF-A0A6A5T025-F1"   # <- change to any entry_id in df
r = df[df.entry_id == entry].iloc[0]
n_term, c_term, residues, contacts = parse_document(r.document)
print(entry, "| len", r.seq_len, "| contacts", r.contacts_emitted)
print("sequence:", sequence_1letter(n_term, c_term, residues))

## Per-contact degrees (optional)

The `document` records *which* residues are in contact but not the contact
*degree*. Those live in the sibling `summary.json` (from `--summary-out`),
keyed per protein. It is large (~282 MB for 2,000 proteins), so load it only
when you need the degrees:

```python
import json
summary = json.load(open(DOCS_PARQUET.with_name("summary.json")))
by_id = {d["entry_id"]: d for d in summary["documents"]}
by_id["AF-E9I562-F1"]["contacts"][:5]   # each: pos_i / pos_j / resnum / resname / degree / flipped
```